---
## This script will do perform Monte-Carlo "hypothesis/independence testing" for co-occurrence of events
#### 6/29/26
---

In [1]:
# Builds off:
# 4_28_concurrent_trends.ipynb
# 7_24_25_MHWTHWConcurrent.ipynb
# 8_26_revisitconcurrent.ipynb (and .py)

In [2]:
# kernel: pangeo23

In [3]:
# imports
import os
import xarray as xr
import numpy as np
import netCDF4 
import glob
import pandas as pd
import geopandas as gpd
from datetime import datetime
from scipy import stats

In [4]:
# interactive plotting stuff 
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import colors
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.lines import Line2D 

#import matplotlib.dates as mdates
%matplotlib inline
plt.rcParams['figure.figsize'] = 12, 6
#%config InlineBackend.figure_format = 'retina'

import cartopy
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point

In [5]:
def thousand_circular_shuffles_1d(indat, seed, nboot=1000):
    '''
    This takes in a 1-d array of event indices of unknown length L, and generates a 1000 by L array where each entry
    is a version of that timeseries which has been swapped around a random index.

    inputs: indat
    returns: swapped_arr

    6/29/26
    '''
    # set random seed
    rng = np.random.default_rng(seed=seed)
    
    # generate 1000 randim indices for which to "circularly" shuffle the event timeseries in the year around, and do that
    # (write a function for this so that I can do this with 90day blocks)
    
    max_num = len(indat) # maximum indix to choose = length of time dimension of input data
    indices_to_sel = np.array(range(1, max_num-1)) # gen np.array of potential indices to choose from (saying you have to choose at least 1 "in" from the edge of timeseries
    
    random_indices = np.random.choice(indices_to_sel, size=nboot, replace=True) # randomly select 1000 indices (with replacememnt, will double-count since 1k > 365...)
    
    # empty array to save "swapped" data
    swapped_arr = np.zeros(shape=(nboot, max_num)) # dims = [nboot, time]
    
    # swap data around for each of the 1k indices
    for i, randidx in enumerate(random_indices):
    
        # Perform the swap
        swapped_arr[i, :-randidx] = indat[randidx:]
        swapped_arr[i, -randidx:] = indat[:randidx]

    return swapped_arr

def thousand_circular_shuffles_2d(indat, seed, nboot=1000):
    '''
    This takes in a 2-d array of event indices of unknown length L (shape [staid, time/L]), and generates a [staid, time, nboot] array where each entry
    is a version of that timeseries which has been swapped around a random index.

    inputs: indat
    returns: swapped_arr

    6/29/26
    '''
    # set random seed
    rng = np.random.default_rng(seed=seed)
    
    # generate 1000 randim indices for which to "circularly" shuffle the event timeseries in the year around, and do that
    # (write a function for this so that I can do this with 90day blocks)
    
    max_num = indat.shape[1] # maximum indix to choose = length of time dimension of input data
    indices_to_sel = np.array(range(1, max_num-1)) # gen np.array of potential indices to choose from (saying you have to choose at least 1 "in" from the edge of timeseries
    
    random_indices = np.random.choice(indices_to_sel, size=nboot, replace=True) # randomly select 1000 indices (with replacememnt, will double-count since 1k > 365...)
    
    # empty array to save "swapped" data
    swapped_arr = np.zeros(shape=(indat.shape[0], max_num, nboot)) # dims = [staid, time, nboot]
    
    # swap data around for each of the 1k indices
    for i, randidx in enumerate(random_indices):
    
        # Perform the swap
        swapped_arr[:, :-randidx, i] = indat[:, randidx:]
        swapped_arr[:, -randidx:, i] = indat[:, :randidx]

    return swapped_arr

In [6]:
script = 'FigS9_resubmissionJun26_concurrent_independence_test.ipynb'

In [7]:
## TESTS
    ## do the 'split-year' "circular" reshuffling, (shifting BOTH) --> maybe also do holding 1 constant and holding the other constant? 

    ## doing the "circular" reshuffle but within 90-day blocks?

# want to generate counterfactual event co-occurrence time series and trends. 


# also do this for holding THW constant, for holding MHW constant

In [8]:
# dataframe with the stations we are using
df = pd.read_csv('/home/nsiegert/projects/coastal_sst/data/hadisd_stations_using_Expanded.csv')
df = df.drop(['Unnamed: 0'], axis=1)

# convert df into geodataframe for ease of plotting
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(x=df.LON, y=df.LAT))

In [9]:
# open station data
hw_ds = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.1.5deg.marineheatwaves.nc')
thw_ds = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.1.5deg.heatwaves.nc')

In [10]:
# ID "factual" timeseries of MHW, THW, and concurrent
mhw_mask = hw_ds.MHW
thw_mask = thw_ds.THW
con_mhwthwdays = hw_ds.MHW * thw_ds.THW

## analysis

In [11]:
if os.path.exists('/dx02/data/nsiegert/coastal_mhw_data/con_evs_counterfactuals_ds.nc'):
    print('FILES EXIST, NO NEED TO RUN THIS AGAIN!')

FILES EXIST, NO NEED TO RUN THIS AGAIN!


# plots

In [ ]:
# load the datasets

outnames = ['', 'mhwconst_', 'thwconst_']
cf_dict = {}

for i in range(3):

    da_in = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/con_evs_counterfactuals_{}ds.nc'.format(outnames[i])).con

    cf_dict[outnames[i]] = da_in.resample(time='YS').sum() # need to sum the quarterly data to the annual scale when I load it in

# gen global CI's and stuff
for outname in outnames:

    # gen the 50% 97.5 and 02.5% quantiles
    cf_dict[outname]['50'] = cf_dict[outname].sum(dim='staid').quantile(0.5, dim='boot')
    cf_dict[outname]['975'] = cf_dict[outname].sum(dim='staid').quantile(0.975, dim='boot')
    cf_dict[outname]['025'] = cf_dict[outname].sum(dim='staid').quantile(0.025, dim='boot')

In [ ]:
# compute the factual stuff. 
con_mhwthwdays_ann = con_mhwthwdays.sum(dim='staid').resample(time='YS').sum()
con_mhwthwdays_ann_pctchg = (con_mhwthwdays_ann / con_mhwthwdays_ann.sel(time=slice('1990-01-01', '1990-12-31')).mean()) * 100

In [ ]:
# then want to plot: factual vs. counterfactual for my 3 different "scenarios"

In [ ]:
cf_dict['']['50']

In [ ]:
fig, ax = plt.subplots(3, 1, sharex=True, figsize=(10,8))

for i in range(3):
    
    con_mhwthwdays_ann.plot(ax=ax[i], label='Observed')


# ax0: "full" counterfactual shuffling both event types
cf_dict['']['50'].plot(ax=ax[0], label='Counterfactual')
ax[0].fill_between(x=cf_dict['']['50'].time, y1=cf_dict['']['025'], y2=cf_dict['']['975'], color='tab:orange', alpha=0.4)

# ax1: counterfactual only shuffling THW
cf_dict['mhwconst_']['50'].plot(ax=ax[1], label='Counterfactual (MHW Constant)')
ax[1].fill_between(x=cf_dict['mhwconst_']['50'].time, y1=cf_dict['mhwconst_']['025'], y2=cf_dict['mhwconst_']['975'], color='tab:orange', alpha=0.4)

# ax2: counterfactual only shuffling MHW
cf_dict['thwconst_']['50'].plot(ax=ax[2], label='Counterfactual (THW Constant)')
ax[2].fill_between(x=cf_dict['thwconst_']['50'].time, y1=cf_dict['thwconst_']['025'], y2=cf_dict['thwconst_']['975'], color='tab:orange', alpha=0.4)


for i in range(3):
    ax[i].set(xlabel='', ylabel='N. Days', title='')
    ax[i].legend()

plt.suptitle('Comparing Observed and Counterfactual Global-Sum Concurrent Event Counts', fontsize=16)
plt.show()

In [ ]:
# so, this is telling me that on the global scale how I've constructed things,
    # a) the observed increasing trend in CON seems to likely be able to be mostly explained (magnitude-wise) by increasing frequency of MHW and THW
        # events plus random chance of "saturation" / their overlap. 
    # b) the "gap" between "observed" and "counterfactual" timeserieses (on any quantile of estimation) is DECREASING, meaning that they appear to be systematically
        # grouping together less strongly than before?

        # the "residual co-occurrence" (above what chance overlap predicts from my randomization) is NOT increasing. --> Can I ask this at every location as well? 

        # if the gap between observed and "independent"/random counterfactual was growing, you'd say that there was strengthening dependence between the two... huh. 
        # is this true at other locations though? maybe that's a question to ask. 

In [ ]:
fig, ax = plt.subplots()

(cf_dict['']['50'] / con_mhwthwdays_ann).plot(ax=ax, label='Shuffling Both')
(cf_dict['thwconst_']['50'] / con_mhwthwdays_ann).plot(ax=ax, label='THW Constant')
(cf_dict['mhwconst_']['50'] / con_mhwthwdays_ann).plot(ax=ax, label='MHW Constant')

ax.legend()
ax.set(ylabel='Frac.', xlabel='Time', title='Is the Ratio of Counterfactual to Observed CON Events Changing? (global)')

plt.show()

In [ ]:
## compute and compare trend slopes

In [ ]:
## GLOBAL MEAN slopes

In [ ]:
# use polyfit to get predicted yvals
fit1 = con_mhwthwdays_ann.polyfit(dim='time', deg=1)

# 3. Evaluate the fit to get trend line values
trend_line = xr.polyval(con_mhwthwdays_ann.time, fit1.polyfit_coefficients)

fig, ax = plt.subplots()

con_mhwthwdays_ann.plot(ax=ax)
trend_line.plot(ax=ax)

In [ ]:
# counterfactual slopes --> this stuff is easy to implement but the magnitude of slopes and intercepts is wacky - due to time dimension of data?
cf_slopes = cf_dict[''].sum(dim='staid').polyfit(dim='time', deg=1).polyfit_coefficients.sel(degree=1)

In [ ]:
cf_slopes.min()

In [ ]:
cf_slopes.max()

In [ ]:
fit1.polyfit_coefficients.sel(degree=1)

In [ ]:
## why are the polyfit slopes so small?????

In [ ]:
# comparing the counterfaactual to observed slopes. 
plt.hist(cf_slopes)
plt.axvline(fit1.polyfit_coefficients.sel(degree=1))

In [ ]:
## as ablove, just using numpy

In [ ]:
xs = np.array(range(1, 35))

In [ ]:
con_mhwthwdays_annDAT = con_mhwthwdays_ann.data

In [ ]:
slope1, int1 = np.polyfit(x=xs, y=con_mhwthwdays_annDAT, deg=1)

In [ ]:
# can fit 2d, if I have one dataset per COLUMN
slope2, int2 = np.polyfit(x=xs, y=cf_dict[''].data.sum(axis=0), deg=1)

In [ ]:
slope1

In [ ]:
# comparing histogram of 
plt.hist(slope2)
plt.axvline(slope1)

In [ ]:
plt.plot(xs, con_mhwthwdays_ann)
plt.plot(xs, cf_dict[''].data.sum(axis=0).mean(axis=1))
plt.plot(xs, (int1 + (slope1 * xs)))
plt.plot(xs, (np.mean(int2) + (np.mean(slope2) * xs)))

In [ ]:
# look at the initial values of the concurrent data
con_mhwthwdays_ann[:5]

In [ ]:
cf_dict[''].data.sum(axis=0).mean(axis=1)[:5]

In [ ]:
np.polyfit?

In [ ]:
print('hello')

In [ ]:
print('A constant: 2.5 percentile', cf_dict['thwconst_']['025'].values)
print('A constant: 50 percentile',cf_dict['thwconst_']['50'].values)
print('A constant: 97.5 percentile',cf_dict['thwconst_']['975'].values)

In [ ]:
print('B constant: 2.5 percentile', cf_dict['mhwconst_']['025'].values)
print('B constant: 50 percentile',cf_dict['mhwconst_']['50'].values)
print('B constant: 97.5 percentile',cf_dict['mhwconst_']['975'].values)

In [ ]:
# are MHW's "saturating" in the timeseries?
mhw_mask.mean(dim='staid').resample(time='YS').mean().plot()

In [ ]:
# compute annual concurrent days for each station
con_mhwthwdays_ann_bySTA = con_mhwthwdays.resample(time='YS').sum()

In [ ]:
con_mhwthwdays_ann_bySTA_slopes = con_mhwthwdays_ann_bySTA.polyfit(dim='time', deg=1).polyfit_coefficients.sel(degree=1)

In [ ]:
## is this true at other / locations all though? ## ---> THIS ISN'T WORKING YET

In [ ]:
# compute annual concurrent days for each station
con_mhwthwdays_ann_bySTA = con_mhwthwdays.resample(time='YS').sum()

# compute annual median counterfactual estimate for each station
cf_bySTA_meds = cf_dict[''].quantile(0.5, dim='boot')

# ratio
ratio_cf_obs_bysta_t1 = (cf_bySTA_meds.sel(time=slice('2014-01-01', '2023-12-31')).mean(dim='time') / con_mhwthwdays_ann_bySTA.sel(time=slice('2014-01-01', '2023-12-31')).mean(dim='time'))
ratio_cf_obs_bysta_t0 = (cf_bySTA_meds.sel(time=slice('1990-01-01', '1999-12-31')).mean(dim='time') / con_mhwthwdays_ann_bySTA.sel(time=slice('1990-01-01', '1999-12-31')).mean(dim='time'))
    # computing it this way b/c that way ratio's don' blow up with inf's when a given station-year has 0 concurrent events

# delta's
d_ratio_cf_obs_bysta = ratio_cf_obs_bysta_t1 - ratio_cf_obs_bysta_t0

In [ ]:
nan_inf_mask = np.logical_not(np.isnan(d_ratio_cf_obs_bysta) + np.isinf(d_ratio_cf_obs_bysta))

gdf['d_ratio_cf_obs_bysta'] = d_ratio_cf_obs_bysta

In [ ]:
plt.hist(d_ratio_cf_obs_bysta[nan_inf_mask])

In [ ]:
print('hello')

In [ ]:
## is there a seasonal aspect to my event occurrences?

In [ ]:
mhw_mask.sum(dim='staid').groupby('time.dayofyear').mean().plot()

In [ ]:
thw_mask.sum(dim='staid').groupby('time.dayofyear').mean().plot()

In [ ]:
con_mhwthwdays.sum(dim='staid').groupby('time.dayofyear').mean().plot()

In [ ]:
## Ok dang, at least on the global scale, there is seasonality in my events on the global scale

---

# try to implement the 90-day shuffles

## analyze the 92-day rolling window shuffle

In [ ]:
# load the datasets

outnames = ['', 'mhwconst_', 'thwconst_']
cf_92dict = {}

for i in range(3):

    da_in = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/con_evs_92dshuffle_counterfactuals_{}ds.nc'.format(outnames[i])).con

    cf_92dict[outnames[i]] = da_in.resample(time='YS').sum() # need to sum the quarterly data to the annual scale when I load it in

# gen global CI's and stuff
for outname in outnames:

    # gen the 50% 97.5 and 02.5% quantiles
    cf_92dict[outname]['50'] = cf_92dict[outname].sum(dim='staid').quantile(0.5, dim='boot')
    cf_92dict[outname]['975'] = cf_92dict[outname].sum(dim='staid').quantile(0.975, dim='boot')
    cf_92dict[outname]['025'] = cf_92dict[outname].sum(dim='staid').quantile(0.025, dim='boot')

In [ ]:
fig, ax = plt.subplots(3, 1, sharex=True, figsize=(10,8))

for i in range(3):
    
    con_mhwthwdays_ann.plot(ax=ax[i], label='Observed')


# ax0: "full" counterfactual shuffling both event types
cf_92dict['']['50'].plot(ax=ax[0], label='Counterfactual')
ax[0].fill_between(x=cf_92dict['']['50'].time, y1=cf_92dict['']['025'], y2=cf_92dict['']['975'], color='tab:orange', alpha=0.4)

# ax1: counterfactual only shuffling THW
cf_92dict['mhwconst_']['50'].plot(ax=ax[1], label='Counterfactual (MHW Constant)')
ax[1].fill_between(x=cf_92dict['mhwconst_']['50'].time, y1=cf_92dict['mhwconst_']['025'], y2=cf_92dict['mhwconst_']['975'], color='tab:orange', alpha=0.4)

# ax2: counterfactual only shuffling MHW
cf_92dict['thwconst_']['50'].plot(ax=ax[2], label='Counterfactual (THW Constant)')
ax[2].fill_between(x=cf_92dict['thwconst_']['50'].time, y1=cf_92dict['thwconst_']['025'], y2=cf_92dict['thwconst_']['975'], color='tab:orange', alpha=0.4)


for i in range(3):
    ax[i].set(xlabel='', ylabel='N. Days', title='')
    ax[i].legend()

plt.suptitle('Comparing Observed and Counterfactual Global-Sum Concurrent Event Counts \n (92-day window for shuffling)', fontsize=16)
plt.show()

In [ ]:
# compare to make sure the data are not exactly the same...

cf_92dict['']['50'].plot()
cf_dict['']['50'].plot()

In [ ]:
# are the MHW vs. THW constant things providing any difference?

cf_92dict['']['50'].plot()
cf_92dict['mhwconst_']['50'].plot()
cf_92dict['thwconst_']['50'].plot()


# essentially the exact same. I do wonde if I might be making the data incorrectly. 

In [ ]:
fig, ax = plt.subplots()

(cf_92dict['']['50'] / con_mhwthwdays_ann).plot(ax=ax, label='Shuffling Both')
(cf_92dict['thwconst_']['50'] / con_mhwthwdays_ann).plot(ax=ax, label='THW Constant')
(cf_92dict['mhwconst_']['50'] / con_mhwthwdays_ann).plot(ax=ax, label='MHW Constant')

ax.legend()
ax.set(ylabel='Frac.', xlabel='Time', title='Is the Ratio of Counterfactual to Observed CON Events Changing? (global, 92day)')

plt.show()

In [ ]:
(cf_92dict['']['50'] / con_mhwthwdays_ann)[:10].mean()

In [ ]:
(cf_92dict['']['50'] / con_mhwthwdays_ann)[-10:].mean()

## implement the shuffling where I hold MHW or THW event counts at their 1990 level

In [ ]:
if os.path.exists('/dx02/data/nsiegert/coastal_mhw_data/con_evs_counterfactuals_mhwconst_1990counts_ds.nc'):
    print('FILES EXIST, NO NEED TO RUN THIS AGAIN!')

In [ ]:
# load the datasets

outnames = ['mhwconst_', 'thwconst_']
cf_90const_dict = {}

for i in range(2):

    da_in = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/con_evs_counterfactuals_{}1990counts_ds.nc'.format(outnames[i])).con

    cf_90const_dict[outnames[i]] = da_in.resample(time='YS').sum() # need to sum the quarterly data to the annual scale when I load it in

# gen global CI's and stuff
for outname in outnames:

    # gen the 50% 97.5 and 02.5% quantiles
    cf_90const_dict[outname]['50'] = cf_90const_dict[outname].sum(dim='staid').quantile(0.5, dim='boot')
    cf_90const_dict[outname]['975'] = cf_90const_dict[outname].sum(dim='staid').quantile(0.975, dim='boot')
    cf_90const_dict[outname]['025'] = cf_90const_dict[outname].sum(dim='staid').quantile(0.025, dim='boot')

## Plot for supplemental figure and R2R

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10,10), sharex=True)

# ax0: "full" counterfactual shuffling both event types
con_mhwthwdays_ann.plot(ax=ax[0], label='Observed')

cf_dict['']['50'].plot(ax=ax[0], label='Counterfactual')
ax[0].fill_between(x=cf_dict['']['50'].time, y1=cf_dict['']['025'], y2=cf_dict['']['975'], color='tab:orange', alpha=0.4)
ax[0].set(ylabel='Days', title='Observed vs. Counterfactual Concurrent Days if MHW and THW Occurred Independently', xlabel='')
ax[0].legend(loc=(0.01, 0.6))

# inset axis to look at observed vs. counterfactual trend distributions
#inset_ax = inset_axes(ax[0], width="20%", height="30%", loc='center left', borderpad=2)
#inset_ax.hist(slope2, color='tab:orange')
#inset_ax.axvline(slope1, color='tab:blue')
#inset_ax.set(xlabel='Days/Year', ylabel='Count', title='Observed vs. Counterfactual Trends')

# ax1: "ratio" observed to counterfactual (50th pctile)
(con_mhwthwdays_ann/cf_dict['']['50']).plot(ax=ax[1])
ax[1].set(ylabel='Ratio', title='Ratio of Observed to Expected Counterfactual Concurrent Events', xlabel='')

# ax2: holding each event-type constant at 1990 levels
#con_mhwthwdays_ann.plot(ax=ax[2], label='Observed')

cf_90const_dict['mhwconst_']['50'].plot(ax=ax[2], label='Counterfactual (n. MHW Constant at 1990 Levels)', color='tab:green')
ax[2].fill_between(x=cf_90const_dict['mhwconst_']['50'].time, y1=cf_90const_dict['mhwconst_']['025'], y2=cf_90const_dict['mhwconst_']['975'], color='tab:green', alpha=0.4)


cf_90const_dict['thwconst_']['50'].plot(ax=ax[2], label='Counterfactual (n. THW Constant at 1990 Levels)', color='tab:red')
ax[2].fill_between(x=cf_90const_dict['thwconst_']['50'].time, y1=cf_90const_dict['thwconst_']['025'], y2=cf_90const_dict['thwconst_']['975'], color='tab:red', alpha=0.4)

ax[2].set(xlabel='Time', ylabel='Days', title='Contribution of MHW vs. THW Trends to Expected Concurrent Event Counts')
ax[2].legend(loc=(0.01, 0.6))

# subplot letters
ax[0].text(s='a.', x=0.025, y=0.88, transform=ax[0].transAxes, ha='left', fontsize=12)
ax[1].text(s='b.', x=0.025, y=0.88, transform=ax[1].transAxes, ha='left', fontsize=12)
ax[2].text(s='c.', x=0.025, y=0.88, transform=ax[2].transAxes, ha='left', fontsize=12)
ax[0].set_xlim(pd.to_datetime('1989-01-01'), pd.to_datetime('2024-01-01'))

plt.savefig('/home/nsiegert/projects/coastal_sst/plots/grl_resubmission_6.2026/FigSX_coastalsst_grl_6_26_concurrent_counterfactuals.png', format='png', bbox_inches='tight')

plt.show()